# 01 — Data Cleaning

This notebook takes the raw Online Retail II export and turns it into a clean, analysis-ready dataset. **Input:** `data/raw/online_retail_II.csv` (541,910 line items, Dec 2010–Dec 2011, UK-based online retailer). It resolves data quality issues (duplicates, bad-debt adjustments, internal stock write-offs, missing values, dtype mismatches), flags cancelled and zero-value orders, and exports two processed datasets — `revenue_df.csv` (full transaction set) and `customer_df.csv` (customer-attributed subset) — that every notebook downstream builds on.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding='ISO-8859-1')

In [ ]:
df.shape

(541910, 8)

In [ ]:
df.describe()

,Quantity,Price,Customer ID
count,541910.000000,541910.000000,406830.000000
mean,9.552234,4.611138,15287.684160
std,218.080957,96.759765,1713.603074
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  str    
 1   StockCode    541910 non-null  str    
 2   Description  540456 non-null  str    
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  str    
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [ ]:
df.sample(15)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
418428,572721,84879,ASSORTED COLOUR BIRD ORNAMENT,16,10/25/11 15:07,1.69,16201.0,United Kingdom
496499,578344,84536B,FAIRY CAKES NOTEBOOK A7 SIZE,1,11/24/11 9:21,0.83,NaN,United Kingdom
439885,574481,22609,PENS ASSORTED SPACEBALL,36,11/4/11 12:45,0.19,18022.0,United Kingdom
536792,581219,85049C,ROMANTIC PINKS RIBBONS,1,12/8/11 9:28,2.46,NaN,United Kingdom
104518,545186,22754,SMALL RED BABUSHKA NOTEBOOK,2,2/28/11 15:05,0.85,17841.0,United Kingdom
485821,577689,23298,SPOTTY BUNTING,3,11/21/11 11:29,4.95,17664.0,United Kingdom
534144,581134,23576,SNACK TRAY RED VINTAGE DOILY,1,12/7/11 13:12,1.95,16368.0,United Kingdom
13355,537434,21379,CAMPHOR WOOD PORTOBELLO MUSHROOM,2,12/6/10 16:57,2.51,NaN,United Kingdom
363022,568531,23389,SPACEBOY MINI BACKPACK,2,9/27/11 13:49,4.15,16713.0,United Kingdom
461561,575947,84509c,SET OF 4 POLKADOT PLACEMATS,1,11/13/11 11:50,7.46,NaN,United Kingdom


→ There are some stockcodes like C543974 which have c prefix and their (quantity<0 and price>0) which are cancled invoices that should not be included in total revenue.

In [ ]:
df[df['Invoice'].astype(str).str.contains('[A-Za-z]', regex=True, na=False)].value_counts(['Invoice'])


Invoice
C570867    101
C560540     57
C548460     45
C560855     41
C538341     39
          ... 
C581463      1
C581470      1
C581484      1
C581499      1
C581568      1
Name: count, Length: 3839, dtype: int64

check what types of prefixs we do have for invoices

In [ ]:
df[~df['Invoice'].str.isnumeric()]['Invoice'].str[0].value_counts()

Invoice
C    9288
A       3
Name: count, dtype: int64

→ There are some stockcodes like A563185 which have a prefix mean that the bad debt has been adjusted, they are not included in total revenue

In [ ]:
df[df['Invoice'].astype(str).str.startswith('A', na=False)].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
299982,A563185,B,Adjust bad debt,1,8/12/11 14:50,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,8/12/11 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/11 14:52,-11062.06,NaN,United Kingdom


In [ ]:
df[df['Invoice'].str.startswith('C', na=False)]['Quantity'].describe()

count     9288.000000
mean       -29.885228
std       1145.786965
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

→ C-invoices max is -1 which we gives us the hint that all cancelations are negative which is logical.

In [ ]:
unflagged_cancellations = df[(df['Quantity'] < 0) & (df['Price'] < 0) & (~df['Invoice'].astype(str).str.startswith(('C', 'A'), na=False))]
unflagged_cancellations

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country


→ We want to check if there are any invoices numbers that are cancled invoices bt have not been flagged \
be ebarati they are canceled bt have not been flagged canceled



In [ ]:
unflagged_negative_qty = df[(df['Quantity'] < 0) & (df['Price'] == 0)]
unflagged_negative_qty

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/10 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,12/3/10 15:30,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,12/7/11 18:36,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,12/7/11 18:38,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,12/7/11 18:38,0.0,NaN,United Kingdom
536910,581226,23090,missing,-338,12/8/11 9:56,0.0,NaN,United Kingdom


In [ ]:
zero_price_positive_qty = df[(df['Price'] == 0) & (df['Quantity'] > 0)]
print(f"Rows with Price=0, Quantity>0: {len(zero_price_positive_qty)}")
print(f"  - no Customer ID :   {zero_price_positive_qty['Customer ID'].isna().sum()}")
print(f"  - has Customer ID :  {zero_price_positive_qty['Customer ID'].notna().sum()}")

Rows with Price=0, Quantity>0: 1179
  - no Customer ID :   1139
  - has Customer ID :  40


there are some gifts given to customers --> 40 invoices\
also there are some invoices that were sold items with zero price with no cutId --> likely stock adjustments? not sure

In [ ]:
promotional_invoices = df[(df['Quantity'] > 0) & (df['Price'] == 0)]
promotional_invoices.sample(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
14385,537534,21912,VINTAGE SNAKES & LADDERS,1,12/7/10 11:48,0.0,NaN,United Kingdom
74656,542514,22866,NaN,1,1/28/11 12:08,0.0,NaN,United Kingdom
104405,545176,22363,GLASS JAR MARMALADE,1,2/28/11 14:19,0.0,NaN,United Kingdom
382665,569931,37370,NaN,18,10/6/11 17:50,0.0,NaN,United Kingdom
73934,542394,84452,NaN,65,1/27/11 15:11,0.0,NaN,United Kingdom


40 invoices which items which were given for free

In [ ]:
a_invoices = df[df['Invoice'].str.startswith('A', na=False)]
c_invoices = df[df['Invoice'].str.startswith('C', na=False)]

In [ ]:
df.isna().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

In [ ]:
df.duplicated().sum()

5268

In [ ]:
df.nunique()

Invoice        25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
Price           1630
Customer ID     4372
Country           38
dtype: int64

## Summary of issues found

1 - invoice date is str it should become datetime\
2 - customer id is float while we don't need floating points\
3 - customerId is null for 135,080 rows — kept for revenue analysis bt dropped for customer analysis\
4 - There are negative values for quantity + price --> all the negative values are cancelations \
5 - Description is null for 1,454 rows\
6 - Quantity has negative values (min -80995),returns/cancellations or stock adjustments\
7 - There are 5,268 duplicate rows \
8 - There are 1,179 rows with Price = 0 and Quantity > 0 (not caught by the existing negative-quantity write-off filter) — 1,139 are internal notes with no Customer ID, 40 are real customer orders receiving a free/promotional item

## Cleaning steps applied
1. Dropped exact duplicates
2. Dropped 'A' (bad-debt) invoices and unflagged negative qty which were lost or broken or ...
3. Converted InvoiceDate → datetime, Customer ID → int
4. Flagged cancellations (`is_cancelled`)
5. Filled missing Description with 'Unknown'
6. Added `total_payments` (Quantity × Price)
7. Flagged the remaining 40 Price=0/Quantity>0 rows with `is_zero_value` (real customer orders, kept but excluded from order/frequency counts downstream)

In [ ]:
df = df.drop_duplicates()

5268 rows dropped

In [ ]:
df = df.drop(a_invoices.index)

removed the invocies which were bad depth

### Zero-value rows (Price = 0, Quantity > 0)
Positive-quantity side of the same issue as `non_c_returns` above. No Customer ID → internal note, drop. Has a Customer ID → real free item on a real order, keep and flag.

In [ ]:
df = df.drop(unflagged_negative_qty.index)

removed lost broken , check adjustments in invoices


In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%y %H:%M') # keep full datetime (hour/minute) for hourly-level analysis

In [ ]:
df['Customer ID'] = df['Customer ID'].astype('Int64')

→ Int64 (nullable) instead of int64 to keep the remaining nulls without erroring.

In [ ]:
df['is_cancelled'] = df['Invoice'].str.startswith('C', na=False)

In [ ]:
df['is_zero_value'] = (df['Price'] == 0) & (df['Quantity'] > 0)
df['is_zero_value'].sum()

40

In [ ]:
df['Description'] = df['Description'].fillna('Unknown')

In [ ]:
df.isna().sum()


Invoice               0
StockCode             0
Description           0
Quantity              0
InvoiceDate           0
Price                 0
Customer ID      132564
Country               0
is_cancelled          0
is_zero_value         0
dtype: int64

In [ ]:
df.duplicated().sum()

0

No missing Description, no duplicates. Remaining Customer ID nulls are by design (guest orders, kept for revenue-level analysis).

### total payment
`Quantity × Price` per row — computed once here so it's already in both processed CSVs, instead of recreating it in every downstream notebook.

In [ ]:
df['total_payment'] = df['Quantity'] * df['Price']

In [ ]:
revenue_df = df.copy()
customer_df = df.dropna(subset=['Customer ID'])

revenue_df.to_csv('../data/processed/revenue_df.csv', index=False)
customer_df.to_csv('../data/processed/customer_df.csv', index=False)

## Takeaways

The raw export needed real cleanup before it could support any downstream analysis: 5,268 exact duplicate rows (~1%), 3 bad-debt adjustment invoices, and 1,336 internal stock write-offs (negative-quantity, Price = 0, no Customer ID) all had to be identified and removed — none represent real customer transactions, and leaving them in would have inflated revenue and return figures.

A second, related issue surfaced separately: 1,174 rows with Price = 0 but *positive* Quantity — the write-off filter above only catches the negative-quantity side. 1,134 of these have no Customer ID (same internal-adjustment pattern, dropped); the remaining 40 have a real Customer ID — genuine free/promotional items on real orders, so they're kept but flagged `is_zero_value` so order/frequency counts downstream don't mistake them for a real purchase.

The remaining ~25% of rows with no Customer ID aren't a data quality problem — they're guest checkouts, real revenue that just can't be attributed to a specific customer. That's why the pipeline exports two files rather than one: `revenue_df` (everything, for revenue-level KPIs) and `customer_df` (customer-attributed only, for anything requiring a Customer ID).

Cancelled orders (`is_cancelled`, ~9,288 invoices) were flagged, not dropped, so downstream notebooks can compute Net Sales and return rate rather than losing that signal entirely.